# 10 · 🔥 Chaos Lab: Killing a NodeManager Mid-Job (Case C)

**Theory**: docs/02-rdds-lineage-partitions.md ("Lineage: Fault Tolerance Without Replication")

**Prerequisite**: `make up-hadoop` still running, with **both**
`nodemanager1` and `nodemanager2` healthy (`make status`).

We run a deliberately slow job in a background thread, kill one
NodeManager while it's mid-flight, and watch YARN reschedule the lost
Tasks onto the surviving NodeManager — recomputing only what was lost, via
lineage, exactly as docs/02 describes. This mirrors the `restart-datanode3`
demo from `cdn-hadoop-lab`, but for the *processing* engine instead of
storage.

In [ ]:
import subprocess
import sys
import threading
import time

sys.path.insert(0, "../scripts")
from lab_utils import get_yarn_session, layer_path
from pyspark.sql.functions import col

spark = get_yarn_session("10-chaos-lab")
vendas = spark.read.parquet(layer_path("hdfs", "bronze", "vendas"))
print(f"Working set: {vendas.count():,} rows across HDFS")

## A deliberately slow job

A tiny sleep per row (via a Python UDF — yes, the slow kind from docs/05,
used here *on purpose* to stretch out execution time) turns a normally
instant aggregation into a job that takes long enough to interrupt.

In [ ]:
import time as _time

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType


@udf(returnType=DoubleType())
def slow_identity(valor):
    _time.sleep(0.01)  # 10ms/row -> deliberately slow, for demo purposes only
    return valor


result_holder = {}


def run_slow_job():
    start = _time.perf_counter()
    total = (
        vendas.limit(20_000)
        .withColumn("valor_slow", slow_identity(col("valor")))
        .agg({"valor_slow": "sum"})
        .collect()
    )
    result_holder["seconds"] = _time.perf_counter() - start
    result_holder["total"] = total[0][0]


job_thread = threading.Thread(target=run_slow_job)
job_thread.start()
print("Job launched in the background — check http://localhost:8088 for the Application.")

## Kill a NodeManager while the job is running

Give the job a few seconds to actually start executing on both
NodeManagers, then kill one.

In [ ]:
time.sleep(8)
print("💥 Killing nodemanager1 mid-job...")
subprocess.run(["docker", "kill", "nodemanager1"], check=True)
print("nodemanager1 is down. Watch the ResourceManager UI: the Application")
print("should keep running, rescheduling nodemanager1's lost tasks onto")
print("nodemanager2 — no data was lost, because Spark recomputes lost")
print("partitions from lineage (docs/02), it doesn't need a backup copy.")

In [ ]:
job_thread.join()
print(f"Job finished in {result_holder['seconds']:.1f}s despite losing a NodeManager mid-flight.")
print(f"Result: {result_holder['total']:.2f}")

## Bring nodemanager1 back

```bash
docker start nodemanager1
```

Run the cell below to confirm it rejoins the cluster (check
http://localhost:8088/cluster/nodes for both NodeManagers listed as
`RUNNING` again).

In [ ]:
subprocess.run(["docker", "start", "nodemanager1"], check=True)
print("nodemanager1 restarted — give it ~30s, then check http://localhost:8088/cluster/nodes")
spark.stop()